In [1]:
import torch
import numpy as np
from torch import nn
import matplotlib.pyplot as plt
from nuscenes.nuscenes import NuScenes,NuScenesExplorer
data_path = 'D:/Projects/Minor Project/Datasets/raw/nuScenes'

In [8]:
import sys
import os

# 1. Get the current directory (Minor Project/Notebook)
# 2. Get the parent directory (Minor Project)
root_path = os.path.abspath(os.path.join(os.getcwd(), ".."))

# 3. Add that parent directory to sys.path
if root_path not in sys.path:
    sys.path.append(root_path)

In [9]:
from Models.dataloader import CustomDataset,DataLoader
nusc = NuScenes('v1.0-mini',dataroot=data_path,verbose=True)
explorer = NuScenesExplorer(nusc)

dataset = CustomDataset(nusc,explorer,nusc.sample)
train_loader = DataLoader(dataset,batch_size=4,shuffle=True)

Loading NuScenes tables for version v1.0-mini...
Loading nuScenes-lidarseg...
32 category,
8 attribute,
4 visibility,
911 instance,
12 sensor,
120 calibrated_sensor,
31206 ego_pose,
8 log,
10 scene,
404 sample,
31206 sample_data,
18538 sample_annotation,
4 map,
404 lidarseg,
Done loading in 1.148 seconds.
Reverse indexing ...
Done reverse indexing in 0.2 seconds.


In [10]:
class PositionalEmbeddings(nn.Module):
    def __init__(self,num_tokens = 4096,embedding_dim = 256):
        super().__init__()
        self.pos_embedding = nn.Parameter(torch.randn(1,num_tokens,embedding_dim) * 0.02)

    def forward(self,x):
        return x + self.pos_embedding
    
class TransformerBlock(nn.Module):
    def __init__(self, embed_dim=256, nhead=8):
        super().__init__()
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=nhead, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)

    def forward(self, x):
        return self.transformer(x)

In [11]:
tensor = torch.randn(1,256,64,64)
print(tensor.flatten(2).shape)

reshape = tensor.reshape(1,256,64,64)
print(reshape == tensor)

torch.Size([1, 256, 4096])
tensor([[[[True, True, True,  ..., True, True, True],
          [True, True, True,  ..., True, True, True],
          [True, True, True,  ..., True, True, True],
          ...,
          [True, True, True,  ..., True, True, True],
          [True, True, True,  ..., True, True, True],
          [True, True, True,  ..., True, True, True]],

         [[True, True, True,  ..., True, True, True],
          [True, True, True,  ..., True, True, True],
          [True, True, True,  ..., True, True, True],
          ...,
          [True, True, True,  ..., True, True, True],
          [True, True, True,  ..., True, True, True],
          [True, True, True,  ..., True, True, True]],

         [[True, True, True,  ..., True, True, True],
          [True, True, True,  ..., True, True, True],
          [True, True, True,  ..., True, True, True],
          ...,
          [True, True, True,  ..., True, True, True],
          [True, True, True,  ..., True, True, True],
      

In [ ]:
class DepthSAMFusion(nn.Module):
    def __init__(self):
        super().__init__()

        self.pos_encoder = PositionalEmbeddings()
        self.fusion_transformer = TransformerBlock()

        self.output_conv = nn.Conv2d(256,256,kernel_size=1)

    def forward(self,img_feature,depth_feature):
        # Features has shape [B,256,64,64]
        B,C,H,W = img_feature.shape

        # Flatten should start from dimension 2 and we get result in shape [B,256,4096]
        # then we reshape it to [B,4096,256]
        img_tokens = img_feature.flatten(2).permute(0,2,1)
        depth_tokens = depth_feature.flatten(2).permute(0,2,1)      

        img_tokens = self.pos_encoder(img_tokens)
        depth_tokens = self.pos_encoder(depth_tokens)

        fused_tokens = self.fusion_transformer(img_tokens+depth_tokens)

        fused_grid = fused_tokens.permute(0,2,1).reshape(B,C,H,W)

        return self.output_conv(fused_grid)




In [15]:
pos_encoder = PositionalEmbeddings()
rand = torch.randn(1,4096,256)
print(rand)
embedded = pos_encoder(rand)
print(embedded)

tensor([[[-0.8551,  0.3187, -0.5241,  ..., -1.7625,  1.2479,  0.3126],
         [-0.6603, -0.8855, -1.0654,  ..., -2.0396,  0.5145,  1.1584],
         [-0.1402, -0.5442, -0.6054,  ..., -0.3395,  0.2095, -0.1544],
         ...,
         [ 1.1855,  1.6886, -1.2267,  ...,  1.8664,  0.5660, -0.0043],
         [ 0.3930, -0.6610, -1.3102,  ..., -1.7169, -0.2094, -0.0440],
         [-0.4071, -0.9214, -0.2666,  ..., -0.6712,  0.1957,  0.0830]]])
tensor([[[-0.8354,  0.3172, -0.4789,  ..., -1.7494,  1.2201,  0.3299],
         [-0.6755, -0.8621, -1.0587,  ..., -2.0461,  0.4844,  1.1474],
         [-0.1149, -0.5445, -0.5996,  ..., -0.3711,  0.2389, -0.1566],
         ...,
         [ 1.1957,  1.7206, -1.2447,  ...,  1.8831,  0.5318, -0.0055],
         [ 0.3574, -0.6510, -1.2915,  ..., -1.7285, -0.2035, -0.0429],
         [-0.4398, -0.9179, -0.3036,  ..., -0.6688,  0.2101,  0.0653]]],
       grad_fn=<AddBackward0>)
